# Homer Transformer

Ce notebook entraîne un Transformer encodeur-décodeur, codé à la main avec PyTorch, pour qu'il réponde comme
Homer Simpson. On lui donne une réplique, par exemple `Marge: Tu as fini ?`, et il écrit la réponse d'Homer.

L'encodeur lit la réplique en entier. Le décodeur écrit la réponse token par token : il ne voit que ce qu'il a
déjà écrit (masque causal) et va chercher ce dont il a besoin dans la sortie de l'encodeur (cross-attention).

Côté données, on part des dialogues en français de 473 épisodes des Simpsons. Ça ne fait qu'environ 18 000
réponses d'Homer, ce qui est trop peu pour apprendre à construire des phrases correctes. On entraîne donc en
deux temps : d'abord un pré-entraînement sur des sous-titres de films et de séries, pour que le modèle apprenne
le français parlé, puis un fine-tuning sur les répliques d'Homer pour qu'il prenne son style.

Le notebook suit cet ordre : le modèle, le tokenizer, les données, les deux phases d'entraînement, et pour finir
la discussion avec Homer.

## Installation

Sur Colab, il faut d'abord choisir un GPU : Exécution > Modifier le type d'exécution > GPU T4.

La cellule suivante installe `tiktoken`, monte Google Drive et télécharge les dialogues des Simpsons depuis
Hugging Face. Les modèles sont sauvegardés dans le Drive parce que Colab efface tous ses fichiers quand la
session se coupe. En local, la cellule se contente de télécharger le dataset s'il n'est pas déjà là.

In [ ]:
import sys
import urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q tiktoken
    from google.colab import drive
    drive.mount("/content/drive")   # le Drive apparaît dans /content/drive/MyDrive

dataset_file = Path("data/Simpsons/dialogues.jsonl")
if not dataset_file.exists():
    dataset_file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://huggingface.co/datasets/Romaindcrl/Simpsons/resolve/main/dialogues.jsonl"
    urllib.request.urlretrieve(url, dataset_file)
print("dataset :", dataset_file, f"({dataset_file.stat().st_size / 1e6:.1f} Mo)")

In [ ]:
import json
import random
import time
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq
import tiktoken
import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Head(nn.Module):
    """Une tête d'attention : self-attention si context est None, cross-attention sinon."""

    def __init__(self, n_embd: int, head_size: int, dropout: float) -> None:
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, context=None):

        source = x if context is None else context

        q = self.query(x)          # les questions viennent toujours de x (celui qui lit)
        k = self.key(source)       # les étiquettes viennent de la source
        v = self.value(source)     # les informations aussi

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T_x, T_source)
        if mask is not None:
            wei = wei.masked_fill(mask == 0, float("-inf"))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ v                                         # (B, T_x, head_size)


class MultiHeadAttention(nn.Module):
    """Plusieurs têtes en parallèle, concaténées puis projetées."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.heads = nn.ModuleList(Head(n_embd, n_embd // n_head, dropout) for _ in range(n_head))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, context=None):
        return self.dropout(self.proj(torch.cat([h(x, mask, context) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    """MLP appliqué à chaque position indépendamment."""

    def __init__(self, n_embd: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    """Bloc Transformer pre-norm : attention puis feed-forward, avec résiduels."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.sa(self.ln1(x), mask)
        return x + self.ffwd(self.ln2(x))


class GPTLanguageModel(nn.Module):
    """Transformer générique : décodeur avec un masque causal, encodeur sans masque."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """Renvoie les logits `(B, T, vocab_size)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x, mask)
        return self.lm_head(self.ln_f(x))



In [ ]:
def causal_mask(T: int, device=None) -> torch.Tensor:
    """(T, T) : chaque position ne voit qu'elle-même et le passé (décodeur)."""
    return torch.tril(torch.ones(T, T, device=device, dtype=torch.bool))


def padding_mask(pad: torch.Tensor) -> torch.Tensor:
    """pad : (B, T) avec 1 = vrai token, 0 = remplissage -> (B, 1, T) : personne ne regarde le remplissage."""
    return pad.bool()[:, None, :]


class Encoder(nn.Module):
    """Chaque position voit toute la séquence. Renvoie un vecteur par position."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        # pas de lm_head : l'encodeur ne prédit rien, il produit des représentations

    def forward(self, idx: torch.Tensor, pad: torch.Tensor | None = None) -> torch.Tensor:
        """idx : (B, T). pad optionnel : (B, T), 1 = vrai token, 0 = remplissage.
        Renvoie `(B, T, n_embd)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

        # pas de masque causal : seulement le padding, s'il y en a
        mask = padding_mask(pad) if pad is not None else None

        for block in self.blocks:
            x = block(x, mask)
        return self.ln_f(x)

In [13]:
class DecoderBlock(nn.Module):
    """Bloc décodeur pre-norm : self-attention masquée, cross-attention, puis feed-forward."""

    def __init__(self, n_embd: int, n_head: int, dropout: float) -> None:
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, dropout)    # self-attention
        self.ca = MultiHeadAttention(n_embd, n_head, dropout)    # cross-attention
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, enc_out, self_mask=None, cross_mask=None):
        # 1. le texte du décodeur se regarde lui-même (sans voir le futur)
        x = x + self.sa(self.ln1(x), self_mask)

        # 2. le décodeur va chercher l'information dans la sortie de l'encodeur
        x = x + self.ca(self.ln2(x), cross_mask, context=enc_out)

        # 3. chaque position réfléchit seule
        return x + self.ffwd(self.ln3(x))

In [14]:
mha = MultiHeadAttention(n_embd=32, n_head=4, dropout=0.0)
x = torch.randn(2, 5, 32)
enc = torch.randn(2, 9, 32)
print(mha(x).shape)               # torch.Size([2, 5, 32])
print(mha(x, context=enc).shape)

torch.Size([2, 5, 32])
torch.Size([2, 5, 32])


## Le décodeur et le modèle complet

Le décodeur empile des `DecoderBlock`, comme l'encodeur empile des `Block`. Il a en plus une `lm_head`, qui
transforme chaque vecteur en un score par token du vocabulaire (les logits).

L'encodeur, le décodeur et la `lm_head` partagent la même table d'embeddings (le *weight tying*, comme dans
GPT-2). Les trois font le même travail dans un sens ou dans l'autre, token vers vecteur ou vecteur vers token,
donc ça n'a pas beaucoup de sens d'en apprendre trois. On divise par trois les paramètres de cette partie, et
chaque ligne de la table est mise à jour dès que son token apparaît, qu'il soit lu ou écrit.

In [ ]:
class Decoder(nn.Module):
    """Écrit la réponse : chaque position ne voit que le passé et regarde la sortie de l'encodeur."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(DecoderBlock(n_embd, n_head, dropout) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)   # un score par token

    def forward(self, idx, enc_out, enc_pad=None):
        """idx : (B, T) la réponse en cours. enc_out : (B, T_src, n_embd). enc_pad : (B, T_src).
        Renvoie les logits `(B, T, vocab_size)`."""
        _, T = idx.shape
        x = self.token_embedding_table(idx) + self.position_embedding_table(torch.arange(T, device=idx.device))

        # Pas besoin de masquer le padding de la réponse : il est toujours à la fin,
        # donc le masque causal le cache déjà aux vrais tokens.
        self_mask = causal_mask(T, idx.device)
        # En revanche, on ne doit pas regarder le padding de la question.
        cross_mask = padding_mask(enc_pad) if enc_pad is not None else None

        for block in self.blocks:
            x = block(x, enc_out, self_mask, cross_mask)
        return self.lm_head(self.ln_f(x))


class HomerTransformer(nn.Module):
    """L'encodeur lit la réplique, le décodeur écrit la réponse."""

    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float) -> None:
        super().__init__()
        self.block_size = block_size
        self.encoder = Encoder(vocab_size, block_size, n_embd, n_head, n_layer, dropout)
        self.decoder = Decoder(vocab_size, block_size, n_embd, n_head, n_layer, dropout)

        # Weight tying : le décodeur et la lm_head réutilisent la table de l'encodeur.
        # Ce n'est pas une copie, c'est le même objet en mémoire.
        self.decoder.token_embedding_table = self.encoder.token_embedding_table
        self.decoder.lm_head.weight = self.encoder.token_embedding_table.weight

        # Petites valeurs au départ (std 0.02, comme GPT-2). Avec l'initialisation par défaut (std 1),
        # la lm_head partagée donnerait des scores énormes au début de l'entraînement.
        for table in (self.encoder.token_embedding_table,
                      self.encoder.position_embedding_table, self.decoder.position_embedding_table):
            nn.init.normal_(table.weight, std=0.02)

    def forward(self, src, src_pad, tgt_in):
        enc_out = self.encoder(src, src_pad)              # (B, T_src, n_embd)
        return self.decoder(tgt_in, enc_out, src_pad)     # (B, T_tgt, vocab_size)

## Hyperparamètres

Tous les réglages sont regroupés ici. En local sur CPU, si c'est trop lent, on peut baisser `N_EMBD`,
`N_LAYER` ou `MAX_LEN`.

`N_SUBTITLE_PAIRS` fixe le nombre de paires de sous-titres utilisées pour le pré-entraînement. Chaque million
de paires prend environ 2 Go de RAM et Colab gratuit en a à peu près 12 : au-delà d'un million et demi, la
session risque de planter.

`PRETRAIN` indique si on refait le pré-entraînement (`True`) ou si on reprend le `homer_pretrain.pt` déjà
sauvegardé (`False`), par exemple quand Colab s'est déconnecté pendant le fine-tuning.

In [ ]:
# Données
DATA_PATH = Path("data/Simpsons/dialogues.jsonl")
MAX_LEN = 64            # longueur max (en tokens) d'une réplique et d'une réponse
VAL_RATIO = 0.1         # part des paires gardée pour la validation

# Sous-titres pour le pré-entraînement
SUBTITLES_URL = ("https://huggingface.co/datasets/sentence-transformers/parallel-sentences-opensubtitles"
                 "/resolve/main/en-fr/train-00000-of-00002.parquet")   # 200 Mo
SUBTITLES_PATH = Path("data/opensubtitles_fr.parquet")
N_SUBTITLE_PAIRS = 300_000
MIN_TOKEN_COUNT = 5     # en dessous, un token des sous-titres est remplacé par UNK

# Modèle
N_EMBD = 256            # taille des vecteurs
N_HEAD = 4              # N_EMBD doit être divisible par N_HEAD
N_LAYER = 4             # nombre de blocs dans l'encodeur et dans le décodeur
DROPOUT = 0.1

# Entraînement
BATCH_SIZE = 64
PRETRAIN = True         # False : on reprend homer_pretrain.pt au lieu de refaire la phase 1
PRETRAIN_EPOCHS = 3
PRETRAIN_LR = 3e-4
FINETUNE_EPOCHS = 5     # peu de données : le modèle apprend vite par cœur
FINETUNE_LR = 1e-4      # plus petit, pour ne pas effacer ce qu'il a appris sur les sous-titres

# Sur Colab on sauvegarde dans le Drive, en local dans le dossier du projet
SAVE_DIR = Path("/content/drive/MyDrive") if IN_COLAB else Path(".")
PRETRAIN_PATH = SAVE_DIR / "homer_pretrain.pt"       # modèle à la fin de la phase 1
CHECKPOINT_PATH = SAVE_DIR / "homer_transformer.pt"  # modèle final

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
random.seed(42)
USE_FP16 = device == "cuda"   # float16 sur GPU, nettement plus rapide sur une T4

print("device :", device)
if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

## Tokenizer

On utilise le tokenizer BPE de GPT-2 via `tiktoken`. Il découpe le texte en morceaux fréquents : « donuts »
devient « don » + « uts ». Il a été entraîné sur de l'anglais, donc il coupe les mots français en plus de
morceaux (environ deux tokens par mot), mais ça reste bien mieux qu'un découpage lettre par lettre.

GPT-2 connaît 50 257 tokens, et nos dialogues n'en utilisent qu'une petite partie. Une fois les données
chargées, on construira donc notre propre vocabulaire, plus petit.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

gpt2_ids = tokenizer.encode_ordinary("D'oh! Mmm... donuts")
print(gpt2_ids)
print([tokenizer.decode([i]) for i in gpt2_ids])   # le découpage en morceaux

## Les dialogues des Simpsons

`dialogues.jsonl` contient un épisode par ligne, au format JSON :

```
{"title": "Les secrets d'un mariage réussi", "text": "Marge: Oh, Homer...\nHomer: Prendre un bain au whisky ?\n..."}
```

Dans `text`, chaque ligne est une réplique `Personnage: phrase`, et une ligne vide marque un changement de
scène. Chaque fois qu'Homer répond à un autre personnage dans la même scène, on garde la paire
(réplique, réponse d'Homer) :

```
entrée  : "Marge: Il y a ça. Et il y a aussi les cours d'éducation pour adultes."
réponse : "Oh ! Et en quoi l'é-du-ca-tion peut me rendre plus intelligent ? ..."
```

On laisse le nom du personnage dans l'entrée. Le modèle sait ainsi qui lui parle, et c'est ce qui permet
ensuite de lui écrire `Marge: ...`.

In [ ]:
def load_pairs(path: Path) -> list[tuple[str, str]]:
    """Renvoie les paires (réplique d'un personnage, réponse d'Homer)."""
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:                                  # une ligne = un épisode
            episode = json.loads(line)
            previous = None                             # réplique précédente dans la scène

            for replique in episode["text"].split("\n"):
                if ": " not in replique:                # ligne vide : nouvelle scène
                    previous = None
                    continue

                speaker, words = replique.split(": ", 1)    # "Marge: Bonjour" -> "Marge", "Bonjour"
                speaker, words = speaker.strip(), words.strip()

                # quelqu'un parle et Homer lui répond : on garde la paire
                if speaker == "Homer" and words and previous and not previous.startswith("Homer:"):
                    pairs.append((previous, words))

                previous = f"{speaker}: {words}"
    return pairs


pairs = load_pairs(DATA_PATH)
print(f"{len(pairs)} paires chargées depuis {DATA_PATH}")
print(f"dont {sum(q.startswith('Marge:') for q, _ in pairs)} où Homer répond à Marge")
for question, answer in pairs[:3]:
    print(question, "  ->  Homer:", answer)

## Les sous-titres (pré-entraînement)

Les sous-titres viennent du corpus OpenSubtitles
([version Hugging Face](https://huggingface.co/datasets/sentence-transformers/parallel-sentences-opensubtitles),
source : [OPUS](https://opus.nlpl.eu/)). C'est du français parlé et familier, assez proche du ton des
Simpsons, et les répliques sont dans l'ordre du dialogue :

```
Je vais te sauver, Cortex !
Cortex, avant que je ne te sauve, tu pourrais m'apprendre à nager ?
Avec tout l'air qui est dans ta tête, tu devrais flotter.
```

On associe chaque réplique à celle qui la suit. Il n'y a ni noms de personnages ni séparation entre les films,
donc quelques paires mélangent deux scènes, mais pour apprendre le français ce n'est pas gênant.

In [ ]:
if not SUBTITLES_PATH.exists():
    print("téléchargement des sous-titres (200 Mo)...")
    SUBTITLES_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(SUBTITLES_URL, SUBTITLES_PATH)

# La colonne "non_english" contient le sous-titre français (vide quand il n'y en a pas).
lines = pq.read_table(SUBTITLES_PATH, columns=["non_english"]).column(0).to_pylist()


def subtitle_pairs(lines: list[str], n_pairs: int) -> list[tuple[str, str]]:
    """Paires (réplique, réplique suivante) à partir des lignes de sous-titres."""
    pairs = []
    for line, next_line in zip(lines, lines[1:]):
        line = line.strip().lstrip("- ")              # "- J'ai une idée !" -> "J'ai une idée !"
        next_line = next_line.strip().lstrip("- ")
        if line and next_line:                         # une ligne vide coupe le dialogue
            pairs.append((line, next_line))
        if len(pairs) == n_pairs:
            break
    return pairs


pretrain_pairs = subtitle_pairs(lines, N_SUBTITLE_PAIRS)
del lines                                              # on n'en a plus besoin, ça libère de la RAM
print(f"{len(pretrain_pairs)} paires de sous-titres")
for question, answer in pretrain_pairs[:3]:
    print(question, "  ->  ", answer)

### Le vocabulaire

On ne garde que les tokens GPT-2 présents dans nos données : tous ceux des Simpsons, plus ceux des sous-titres
qui apparaissent au moins `MIN_TOKEN_COUNT` fois. Les plus rares (souvent des noms propres vus une seule fois)
deviennent `UNK`, sinon le vocabulaire regrossirait. Les tokens gardés sont renumérotés à partir de 4, et les
numéros 0 à 3 servent aux tokens spéciaux :

| numéro | token | rôle |
|---|---|---|
| 0 | `PAD` | remplissage, pour que toutes les phrases d'un batch aient la même longueur |
| 1 | `BOS` | début de la réponse, premier token donné au décodeur |
| 2 | `EOS` | fin de la réponse |
| 3 | `UNK` | token inconnu |

`encode` découpe le texte avec tiktoken puis passe des numéros GPT-2 aux nôtres, `decode` fait le chemin
inverse. Le vocabulaire est bien plus petit que les 50 259 tokens de départ, et le modèle aussi.

In [ ]:
# Les tokens GPT-2 présents dans nos données
homer_tokens = {t for question, answer in pairs
                for t in tokenizer.encode_ordinary(question) + tokenizer.encode_ordinary(answer)}
subtitle_counts = Counter(t for question, answer in pretrain_pairs
                          for t in tokenizer.encode_ordinary(question) + tokenizer.encode_ordinary(answer))
frequent_subtitle_tokens = {t for t, count in subtitle_counts.items() if count >= MIN_TOKEN_COUNT}
gpt2_ids = sorted(homer_tokens | frequent_subtitle_tokens)

# Tokens spéciaux en 0..3, nos tokens à partir de 4
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3


def build_vocab(gpt2_ids):
    """Renvoie les deux tables de correspondance : notre numéro -> numéro GPT-2, et l'inverse."""
    to_gpt2 = [None] * len(SPECIAL_TOKENS) + list(gpt2_ids)          # to_gpt2[notre numéro] = numéro GPT-2
    from_gpt2 = {g: i for i, g in enumerate(to_gpt2) if g is not None}
    return to_gpt2, from_gpt2


to_gpt2, from_gpt2 = build_vocab(gpt2_ids)
VOCAB_SIZE = len(to_gpt2)


def encode(text: str) -> list[int]:
    # tiktoken découpe, puis on passe aux numéros de notre vocabulaire (UNK si le token n'y est pas)
    return [from_gpt2.get(t, UNK_ID) for t in tokenizer.encode_ordinary(text)]


def decode(ids: list[int]) -> str:
    # on retire les tokens spéciaux et on repasse en numéros GPT-2 pour que tiktoken écrive le texte
    return tokenizer.decode([to_gpt2[i] for i in ids if i >= len(SPECIAL_TOKENS)])


print(f"vocabulaire : {VOCAB_SIZE} tokens (au lieu de {tokenizer.n_vocab + 2} avec tout GPT-2)")
ids = encode("Marge, je veux une bière.")
print(ids, "->", decode(ids))

### Dataset et DataLoader

Pour chaque paire, `src` est la réplique tokenisée (pour l'encodeur) et `tgt` vaut `[BOS] + réponse + [EOS]`
(pour le décodeur). Pendant l'entraînement, le décodeur reçoit `tgt[:-1]` et doit prédire `tgt[1:]`,
c'est-à-dire le token suivant à chaque position :

```
entrée du décodeur : [BOS]  Woo   -hoo   !
à prédire          : Woo    -hoo  !      [EOS]
```

Les phrases n'ont pas toutes la même longueur, donc `collate` complète les plus courtes avec des `PAD`.
`make_loaders` sépare entraînement et validation et renvoie les deux DataLoaders. On l'appelle une fois pour
les sous-titres et une fois pour Homer.

In [ ]:
class DialogueDataset(Dataset):
    """Les paires (réplique, réponse), tokenisées une seule fois au départ."""

    def __init__(self, pairs: list[tuple[str, str]]) -> None:
        self.examples = []
        for question, answer in pairs:
            src = encode(question)[:MAX_LEN]
            tgt = [BOS_ID] + encode(answer)[:MAX_LEN - 2] + [EOS_ID]   # -2 : on laisse la place à BOS et EOS
            self.examples.append((torch.tensor(src), torch.tensor(tgt)))

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def collate(batch):
    """Complète les séquences du batch avec des PAD pour qu'elles aient toutes la même longueur."""
    srcs, tgts = zip(*batch)
    src = pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)   # (B, T_src)
    tgt = pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)   # (B, T_tgt)
    src_pad = (src != PAD_ID).long()                                   # 1 = vrai token, 0 = remplissage
    return src, src_pad, tgt


def make_loaders(pairs: list[tuple[str, str]]):
    """Sépare les paires en entraînement et validation, et renvoie les deux DataLoaders."""
    pairs = pairs[:]                    # copie, pour ne pas mélanger la liste d'origine
    random.shuffle(pairs)
    n_val = max(1, int(len(pairs) * VAL_RATIO))
    train_loader = DataLoader(DialogueDataset(pairs[n_val:]), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(DialogueDataset(pairs[:n_val]), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    print(f"train : {len(train_loader.dataset)} exemples, val : {len(val_loader.dataset)} exemples")
    return train_loader, val_loader


print("sous-titres :", end=" ")
pretrain_train_loader, pretrain_val_loader = make_loaders(pretrain_pairs)
print("Homer :", end=" ")
homer_train_loader, homer_val_loader = make_loaders(pairs)

src, src_pad, tgt = next(iter(homer_train_loader))
print("src", tuple(src.shape), "| tgt", tuple(tgt.shape))

## Le modèle

Un seul modèle, qui passe par les deux phases d'entraînement.

In [ ]:
# Les réglages sont gardés dans un dict pour être sauvegardés avec les poids.
model_config = dict(
    vocab_size=VOCAB_SIZE, block_size=MAX_LEN, n_embd=N_EMBD,
    n_head=N_HEAD, n_layer=N_LAYER, dropout=DROPOUT,
)
model = HomerTransformer(**model_config).to(device)

# En float16, les gradients très petits peuvent tomber à zéro. Le GradScaler multiplie la loss avant
# backward() puis redivise les gradients. Sur CPU (USE_FP16 = False), il ne fait rien.
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params / 1e6:.1f} M paramètres")

## Fonctions d'entraînement

`compute_loss` passe un batch dans le modèle et calcule la cross-entropy entre ses prédictions et les vrais
tokens. Les positions `PAD` sont ignorées : le modèle n'a pas à prédire le remplissage. Sur GPU, ce calcul
se fait en float16 (`torch.autocast`), ce qui prend deux fois moins de mémoire et va plus vite sur une T4.
Les poids, eux, restent en float32.

`train_one_epoch` parcourt une fois les données d'entraînement en mettant à jour les poids. `evaluate` fait
le même calcul sur la validation, sans toucher aux poids. Si la loss de validation remonte alors que celle
d'entraînement continue de baisser, c'est que le modèle apprend par cœur (overfitting).

In [ ]:
def compute_loss(model, batch):
    src, src_pad, tgt = (t.to(device) for t in batch)
    tgt_in = tgt[:, :-1]     # ce que reçoit le décodeur
    tgt_out = tgt[:, 1:]     # ce qu'il doit prédire, décalé d'un cran

    with torch.autocast(device_type=device, dtype=torch.float16, enabled=USE_FP16):
        logits = model(src, src_pad, tgt_in)              # (B, T, VOCAB_SIZE)
    return F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),                   # (B*T, VOCAB_SIZE)
        tgt_out.reshape(-1),                              # (B*T,)
        ignore_index=PAD_ID,
    )


def train_one_epoch(model, loader, optimizer) -> float:
    model.train()                       # active le dropout
    total = 0.0
    for i, batch in enumerate(loader, 1):
        loss = compute_loss(model, batch)
        optimizer.zero_grad()           # on efface les gradients du batch précédent
        scaler.scale(loss).backward()   # on calcule les nouveaux
        scaler.unscale_(optimizer)      # remis à leur vraie taille avant le clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # évite qu'un batch bizarre fasse exploser les poids
        scaler.step(optimizer)          # mise à jour des poids
        scaler.update()
        total += loss.item()
        if i % 500 == 0:
            print(f"   batch {i}/{len(loader)} | loss {total / i:.3f}")
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader) -> float:
    model.eval()                        # désactive le dropout
    total = 0.0
    for batch in loader:
        total += compute_loss(model, batch).item()
    return total / len(loader)

## Générer une réponse

Le décodeur écrit un token à la fois. On encode la réplique une seule fois, puis on part de `[BOS]` : on demande
le token suivant au décodeur, on l'ajoute, et on recommence jusqu'à `EOS` ou jusqu'à la longueur maximale.

Deux réglages jouent sur le choix du token. Avec une `temperature` en dessous de 1 les réponses sont plus
prudentes, au-dessus elles partent davantage dans tous les sens. `top_k` limite le tirage aux k tokens les
plus probables.

In [ ]:
@torch.no_grad()
def reply(model, question: str, max_new_tokens: int = 50, temperature: float = 0.8, top_k: int = 50) -> str:
    model.eval()
    src = torch.tensor([encode(question)[:MAX_LEN]], device=device)   # (1, T_src)
    src_pad = torch.ones_like(src)                                    # une seule phrase, donc pas de padding
    enc_out = model.encoder(src, src_pad)

    out = [BOS_ID]
    for _ in range(min(max_new_tokens, MAX_LEN - 1)):
        idx = torch.tensor([out], device=device)                     # (1, T) la réponse jusqu'ici
        logits = model.decoder(idx, enc_out, src_pad)[0, -1]         # scores du token suivant
        logits[PAD_ID] = logits[BOS_ID] = logits[UNK_ID] = float("-inf")   # tokens interdits en sortie

        logits = logits / temperature
        kth_best = torch.topk(logits, top_k).values[-1]
        logits[logits < kth_best] = float("-inf")                    # on ne garde que les top_k
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()     # tirage au sort selon les probas

        if next_id == EOS_ID:
            break
        out.append(next_id)
    return decode(out[1:])


print(reply(model, "Marge: Homer, tu as fini ton travail ?"))   # avant l'entraînement, ça n'a aucun sens

## Entraînement

`train` s'occupe d'une phase entière. À chaque epoch on entraîne, on évalue, et on sauvegarde le modèle s'il
a la meilleure loss de validation jusque-là. À la fin, on recharge ce meilleur modèle : la phase suivante part
de lui et non de la dernière epoch.

In [ ]:
TEST_QUESTION = "Marge: Homer, tu as fini ton travail ?"


def train(model, train_loader, val_loader, epochs: int, lr: float, save_path: Path) -> float:
    """Entraîne pendant `epochs` epochs, garde le meilleur modèle dans `save_path` et le recharge à la fin."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_loss = evaluate(model, val_loader)

        saved = ""
        if val_loss < best_val:
            best_val = val_loss
            # poids + réglages + vocabulaire : de quoi reconstruire exactement le même modèle
            torch.save({"config": model_config, "vocab": gpt2_ids, "weights": model.state_dict()}, save_path)
            saved = " (sauvegardé)"

        print(f"epoch {epoch:3d} | train {train_loss:.3f} | val {val_loss:.3f} | {time.time() - start:.1f}s{saved}")
        print("   Homer :", reply(model, TEST_QUESTION))

    model.load_state_dict(torch.load(save_path, map_location=device)["weights"])
    return best_val

### Phase 1 : pré-entraînement sur les sous-titres

C'est la phase la plus longue. Le meilleur modèle est sauvegardé dans `homer_pretrain.pt`. Si Colab se coupe
pendant la phase 2, pas besoin de tout recommencer : on met `PRETRAIN = False` dans les hyperparamètres et
cette cellule recharge le fichier. Il ne faut simplement pas avoir changé entre-temps la taille du modèle ou
les données, sinon le vocabulaire ne correspond plus et la cellule le signale.

In [ ]:
if PRETRAIN:
    best = train(model, pretrain_train_loader, pretrain_val_loader, PRETRAIN_EPOCHS, PRETRAIN_LR, PRETRAIN_PATH)
    print(f"meilleure loss val (sous-titres) : {best:.3f}")
else:
    pretrained = torch.load(PRETRAIN_PATH, map_location=device)
    if pretrained["config"] != model_config or pretrained["vocab"] != gpt2_ids:
        raise ValueError(f"{PRETRAIN_PATH} a été entraîné avec d'autres réglages (taille du modèle ou vocabulaire). "
                         "Remets PRETRAIN = True pour refaire le pré-entraînement.")
    model.load_state_dict(pretrained["weights"])
    print("pré-entraînement repris depuis", PRETRAIN_PATH)

### Phase 2 : fine-tuning sur Homer

On continue avec le même modèle, cette fois sur les réponses d'Homer et avec un learning rate plus petit : il
prend le style d'Homer sans oublier le français appris pendant la phase 1. Comme il y a peu de données, la loss
de validation peut remonter assez vite, mais on garde de toute façon la meilleure epoch.

In [ ]:
best = train(model, homer_train_loader, homer_val_loader, FINETUNE_EPOCHS, FINETUNE_LR, CHECKPOINT_PATH)
print(f"meilleure loss val (Homer) : {best:.3f}")

## Récupérer le modèle (Colab)

Le modèle final est dans le Drive (`homer_transformer.pt`, à la racine de Mon Drive). Le plus fiable pour un
gros fichier est de le télécharger depuis drive.google.com, mais la cellule ci-dessous le fait aussi. Pour
discuter avec Homer en local, il faut le placer dans le dossier du projet.

In [ ]:
if IN_COLAB:
    if CHECKPOINT_PATH.exists():
        from google.colab import files
        files.download(str(CHECKPOINT_PATH))
    else:
        print("Pas encore de modèle : il est sauvegardé à la fin de la première epoch du fine-tuning.")

## Discuter avec Homer

On recharge le meilleur modèle sauvegardé, ce qui suppose d'avoir terminé au moins une epoch de la phase 2.
Le modèle est reconstruit à partir des réglages enregistrés avec les poids, et on remet aussi son vocabulaire
pour que `encode` et `decode` utilisent les mêmes numéros qu'à l'entraînement.

Dans la dernière cellule, on joue Marge : il suffit de taper la phrase, `Marge: ` est ajouté devant. On arrête
avec `quit` ou une ligne vide.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
to_gpt2, from_gpt2 = build_vocab(checkpoint["vocab"])        # vocabulaire du modèle sauvegardé
model = HomerTransformer(**checkpoint["config"]).to(device)   # même taille que le modèle sauvegardé
model.load_state_dict(checkpoint["weights"])

for question in ["Marge: Homer, tu veux une salade ?", "Marge: Homer, la centrale a appelé.", "Marge: Tu m'aimes ?"]:
    print(question)
    print("Homer:", reply(model, question))
    print()

In [26]:
while True:
    question = input("Marge: ")
    if question.strip().lower() in ("", "quit"):
        break
    print("Marge:", question)
    print("Homer:", reply(model, f"Marge: {question}"))

Marge: je t'aimes
Marge: je t'aimes
Homer: Marge, je veux pas me faire perdu parce qu'un jour c'est moi.


KeyboardInterrupt: Interrupted by user